In [ ]:
# Cell 1: GPU確認と環境セットアップ

!nvidia-smi

# 必要なライブラリのインストール
!pip install -q -U peft==0.10.0
!pip install -q -U accelerate==0.30.1
!pip install -q -U transformers datasets bitsandbytes
!pip install -q trl gradio

import os
os.makedirs("logs", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("evaluations", exist_ok=True)

print("✅ セットアップ完了!")

In [ ]:
# Cell 2: 拡張データセット生成

import json
import random
from typing import List, Dict

def generate_enhanced_dataset() -> List[Dict]:
    """高品質な拡張データセットを生成"""
    
    dataset = []
    
    # === カテゴリ1: 技術説明 ===
    technical_data = [
        {
            "input": "Transformerアーキテクチャのセルフアテンションメカニズムについて説明してください",
            "output": "セルフアテンションは、入力シーケンス内の各要素が他のすべての要素との関連性を計算する仕組みです。Query、Key、Valueの3つの行列を使用し、注意重みを計算することで、文脈に応じた表現を学習します。これにより、長距離依存関係を効率的に捉えることができます。"
        },
        {
            "input": "LoRAファインチューニングのメリットは何ですか?",
            "output": "LoRA(Low-Rank Adaptation)は、大規模モデルの一部のパラメータのみを学習する効率的な手法です。主なメリットは: 1) 学習パラメータが大幅に削減され、メモリ使用量が少ない、2) 学習時間が短縮される、3) 元のモデルを保持したまま複数のタスクに対応できる、4) 低リソース環境でも実行可能です。"
        },
        {
            "input": "GPUとCPUの違いを説明してください",
            "output": "GPUとCPUの主な違いは並列処理能力です。CPUは複雑な逐次処理に特化した数個から数十個のコアを持ち、汎用的なタスクに適しています。一方、GPUは数千のコアを持ち、同じ処理を大量のデータに対して並列実行できるため、機械学習や画像処理に最適です。"
        },
        {
            "input": "バッチ正規化とは何ですか?",
            "output": "バッチ正規化(Batch Normalization)は、ニューラルネットワークの学習を安定化・高速化する技術です。各ミニバッチごとに、層の出力を正規化(平均0、分散1に変換)することで、学習率を大きく設定でき、収束が早くなります。また、過学習の抑制効果もあります。"
        },
        {
            "input": "Attention機構の役割は?",
            "output": "Attention機構は、入力データの重要な部分に注目(attend)する仕組みです。機械翻訳では、ターゲット言語の各単語を生成する際に、ソース言語のどの単語が重要かを動的に決定します。これにより、長い文でも文脈を適切に捉えることができます。"
        }
    ]
    
    # === カテゴリ2: コーディング ===
    coding_data = [
        {
            "input": "Pythonでリスト内包表記を使う利点は?",
            "output": "リスト内包表記の主な利点は: 1) コードが簡潔で読みやすくなる、2) 通常のforループより実行速度が速い、3) 1行で条件付きフィルタリングとマッピングが可能、4) Pythonic(Pythonらしい)コードスタイルとして推奨されています。例: [x**2 for x in range(10) if x % 2 == 0]"
        },
        {
            "input": "Pythonでファイルを読み込む方法は?",
            "output": "Pythonでファイルを読み込む基本的な方法は、組み込み関数open()を使用することです。推奨される方法は with文を使ったコンテキストマネージャです。例: with open('file.txt', 'r', encoding='utf-8') as f: content = f.read() この方法では、ファイルが自動的にクローズされ、リソースリークを防げます。"
        },
        {
            "input": "デコレータとは何ですか?",
            "output": "デコレータは、関数やメソッドの動作を変更・拡張するためのPythonの構文です。@記号を使って関数定義の前に記述します。ログ記録、実行時間計測、権限チェックなどの横断的関心事を、元の関数を変更せずに追加できます。例: @login_required や @staticmethod など。"
        },
        {
            "input": "非同期処理(async/await)の使い方は?",
            "output": "async/awaitはPythonの非同期プログラミング構文です。async defで非同期関数を定義し、await でその実行を待ちます。I/O待ち時間が長い処理(Web API呼び出し、ファイル読み書き)で、他のタスクを並行実行できるため効率的です。asyncioライブラリと組み合わせて使用します。"
        },
        {
            "input": "例外処理のベストプラクティスは?",
            "output": "例外処理のベストプラクティス: 1) 具体的な例外クラスをキャッチする(Exception は避ける)、2) try ブロックは最小限に、3) finally で必ずリソースを解放、4) カスタム例外クラスを作成して意味のあるエラー情報を提供、5) エラーログを適切に記録、6) ユーザーにわかりやすいエラーメッセージを表示。"
        }
    ]
    
    # === カテゴリ3: 問題解決 ===
    problem_solving_data = [
        {
            "input": "機械学習モデルが過学習している場合、どのような対策がありますか?",
            "output": "過学習への対策には複数のアプローチがあります: 1) より多くの訓練データを収集する、2) データ拡張(augmentation)を適用、3) Dropout層を追加、4) L1/L2正則化を導入、5) Early Stoppingで学習を早期終了、6) モデルの複雑さを減らす、7) バッチ正規化を使用。validation lossをモニタリングしながら、これらを組み合わせて調整します。"
        },
        {
            "input": "メモリリークを防ぐにはどうすればいいですか?",
            "output": "メモリリークを防ぐ方法: 1) Pythonではwith文でリソース管理、2) 循環参照を避ける(weakrefの使用)、3) 大きなオブジェクトは使用後に明示的に削除(del文)、4) ジェネレータを活用して必要なデータだけをメモリに保持、5) メモリプロファイラ(memory_profiler, tracemalloc)で定期的にチェック、6) PyTorchではtorch.cuda.empty_cache()でGPUメモリを解放。"
        },
        {
            "input": "学習が収束しない場合の対処法は?",
            "output": "学習が収束しない場合の対処法: 1) 学習率を下げる(1/10程度に)、2) 学習率スケジューラを使用、3) バッチサイズを調整、4) 勾配クリッピングを導入、5) 重みの初期化方法を変更、6) 活性化関数を見直す(ReLU → GELU等)、7) オプティマイザを変更(SGD → Adam等)、8) データの正規化を確認。Loss曲線をプロットして傾向を分析することが重要です。"
        },
        {
            "input": "データの不均衡問題にどう対処しますか?",
            "output": "クラス不均衡への対処法: 1) オーバーサンプリング: 少数クラスのデータを複製・拡張、2) アンダーサンプリング: 多数クラスのデータを削減、3) SMOTE等の合成サンプリング手法、4) クラスウェイトの調整(loss関数で少数クラスに高い重み)、5) Focal Lossの使用、6) アンサンブル学習、7) 適切な評価指標の選択(Accuracy → F1-score, AUC等)。"
        },
        {
            "input": "推論速度を改善する方法は?",
            "output": "推論速度改善の方法: 1) モデル量子化(8bit/4bit)でメモリと計算量削減、2) モデル蒸留で小型モデルを作成、3) ONNX形式に変換して最適化、4) バッチ処理で複数サンプルを同時処理、5) キャッシングで同じ入力の再計算を避ける、6) GPU推論を使用、7) TensorRT等の推論エンジンを活用、8) 不要な計算(dropout等)を削除。"
        }
    ]
    
    # === カテゴリ4: 比較 ===
    comparison_data = [
        {
            "input": "PyTorchとTensorFlowの違いを教えてください",
            "output": "PyTorchとTensorFlowの主な違い: PyTorchは動的計算グラフ(Define-by-Run)で直感的なデバッグが可能、研究分野で人気。TensorFlowは静的計算グラフ(Define-and-Run)で本番環境への展開に強く、TensorFlow Liteで軽量化も容易です。最近はTensorFlow 2.xでEager Executionがデフォルトとなり、両者の差は縮小しています。"
        },
        {
            "input": "教師あり学習と教師なし学習の違いは?",
            "output": "教師あり学習は、正解ラベル付きデータで学習し、分類や回帰タスクに使用されます(例: 画像分類、価格予測)。教師なし学習は、ラベルなしデータからパターンを発見し、クラスタリングや次元削減に使用されます(例: 顧客セグメント、異常検知)。教師ありは精度が高いですが大量のラベルデータが必要、教師なしはデータ収集が容易ですが解釈が難しいという特徴があります。"
        },
        {
            "input": "CNNとRNNの違いは?",
            "output": "CNN(畳み込みニューラルネットワーク)は、画像の空間的パターンを捉えるのに特化しており、画像認識、物体検出に使用されます。局所的な特徴を階層的に抽出します。RNN(再帰型ニューラルネットワーク)は、時系列データや文章など、順序のあるデータを扱うのに適しており、自然言語処理、音声認識に使用されます。過去の情報を保持しながら処理できるのが特徴です。"
        },
        {
            "input": "AdamとSGDの違いは?",
            "output": "SGD(確率的勾配降下法)はシンプルで解釈しやすく、適切な学習率で良い結果が得られますが、ハイパーパラメータ調整が難しいです。Adam(Adaptive Moment Estimation)は、各パラメータに適応的な学習率を使用し、モメンタムとRMSpropを組み合わせた手法です。デフォルト設定でも良好に動作し、広く使われていますが、一部のタスクではSGDの方が汎化性能が高い場合があります。"
        },
        {
            "input": "BERTとGPTの違いは?",
            "output": "BERTは双方向Transformerで、文章全体の文脈を両方向から理解します。Masked Language Modelingで事前学習され、分類や質問応答に強いです。GPTは単方向(左から右)のTransformerで、Auto-regressiveに次の単語を予測します。テキスト生成に優れています。BERTはエンコーダーのみ、GPTはデコーダーのみという構造上の違いもあります。"
        }
    ]
    
    # === カテゴリ5: 対話的・曖昧な質問 ===
    conversational_data = [
        {
            "input": "機械学習について教えて",
            "output": "機械学習は、コンピュータがデータから自動的にパターンを学習し、予測や判断を行う技術です。大きく分けて、教師あり学習(正解ラベル付きデータで学習)、教師なし学習(パターン発見)、強化学習(試行錯誤で最適行動を学習)の3種類があります。画像認識、音声認識、推薦システム、自動運転など、様々な分野で活用されています。"
        },
        {
            "input": "AIって何?簡単に教えて",
            "output": "AI(人工知能)は、人間の知的活動をコンピュータで模倣する技術の総称です。具体的には、学習(データから規則を発見)、推論(情報から結論を導出)、認識(画像や音声の理解)などができます。身近な例では、スマホの音声アシスタント、写真の自動タグ付け、YouTubeのおすすめ動画などがAI技術を使用しています。"
        },
        {
            "input": "ディープラーニングって難しい?",
            "output": "ディープラーニングの概念理解は比較的シンプルですが、実装には専門知識が必要です。基本的には、多層のニューラルネットワークがデータから特徴を自動抽出する技術です。最近はPyTorchやTensorFlowなどのフレームワークが充実しており、実装の敷居は下がっています。チュートリアルに沿って実践すれば、初心者でも画像分類などの基本タスクは比較的早く習得できます。"
        },
        {
            "input": "どうやって勉強したらいい?",
            "output": "AI・機械学習の学習ステップ: 1) Pythonの基礎を習得、2) 数学(線形代数、微積分、確率統計)の基本を理解、3) 機械学習の基本概念を学ぶ(Coursera、Udacityの講座がおすすめ)、4) 実際にコードを書いて実験(Kaggleのチュートリアル)、5) 小規模プロジェクトで実践、6) 論文を読んで最新動向を追う。手を動かしながら学ぶことが最も効果的です。"
        },
        {
            "input": "初心者におすすめのプロジェクトは?",
            "output": "初心者におすすめのプロジェクト: 1) 手書き数字認識(MNIST)で画像分類の基礎、2) 映画レビューの感情分析でNLPの基礎、3) 住宅価格予測で回帰問題、4) タイタニック生存予測(Kaggleの定番)、5) 簡単なチャットボット作成。既存のデータセットとチュートリアルが豊富なので、挫折しにくく達成感も得られます。"
        }
    ]
    
    # すべてのカテゴリを結合
    dataset.extend(technical_data)
    dataset.extend(coding_data)
    dataset.extend(problem_solving_data)
    dataset.extend(comparison_data)
    dataset.extend(conversational_data)
    
    return dataset

# データセット生成と保存
enhanced_dataset = generate_enhanced_dataset()

with open('data/training_data_v3.json', 'w', encoding='utf-8') as f:
    json.dump(enhanced_dataset, f, ensure_ascii=False, indent=2)

print(f"✅ 拡張データセット生成完了: {len(enhanced_dataset)}件")
print(f"   保存先: data/training_data_v3.json")

# カテゴリ別の件数確認
print("\n【カテゴリ別データ数】")
print("  技術説明: 5件")
print("  コーディング: 5件")
print("  問題解決: 5件")
print("  比較: 5件")
print("  対話的: 5件")
print(f"  合計: {len(enhanced_dataset)}件")

In [ ]:
# Cell 3: モデルとトークナイザーのロード

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from transformers import BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-v0.1"

# 4bit量子化設定
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("📥 モデルをロード中...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

# トークナイザー設定
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id
model.config.use_cache = False

print("✅ モデルロード完了!")
print(f"   モデル: {MODEL_NAME}")
print(f"   量子化: 4bit NF4")
print(f"   デバイス: {next(model.parameters()).device}")

In [ ]:
# Cell 4: データセットの読み込みと前処理

from datasets import load_dataset

# データセットの読み込み
print("📂 データセットを読み込み中...")
dataset = load_dataset("json", data_files="data/training_data_v3.json")

print(f"✅ データセット読み込み完了")
print(f"   データ数: {len(dataset['train'])}件")
print(f"\n【サンプルデータ】")
print(f"Input: {dataset['train'][0]['input']}")
print(f"Output: {dataset['train'][0]['output'][:100]}...")

# 前処理関数
def preprocess_function(examples):
    """データを'User: ... Bot: ...'形式に変換"""
    inputs = examples["input"]
    outputs = examples["output"]
    text_pairs = [f"User: {inp}\nBot: {out}" for inp, out in zip(inputs, outputs)]
    
    model_inputs = tokenizer(
        text_pairs,
        max_length=512,
        truncation=True,
        padding="max_length",
    )
    
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

# データセットの前処理
print("\n🔄 データを前処理中...")
processed_dataset = dataset["train"].map(preprocess_function, batched=True)

print("✅ 前処理完了!")
print(f"   処理済みデータ数: {len(processed_dataset)}件")

In [ ]:
# Cell 5: LoRA設定の適用

from peft import LoraConfig, get_peft_model

# LoRA設定（高性能版）
lora_config = LoraConfig(
    r=16,                    # ランクを増やして表現力向上（8→16）
    lora_alpha=32,           # スケーリング係数
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # 対象モジュール拡大
    lora_dropout=0.1,        # ドロップアウト
    bias="none",
    task_type="CAUSAL_LM",
)

print("🔧 LoRAを適用中...")
model = get_peft_model(model, lora_config)

# 学習可能パラメータの確認
model.print_trainable_parameters()

print("\n✅ LoRA設定完了!")
print(f"   Rank: {lora_config.r}")
print(f"   Alpha: {lora_config.lora_alpha}")
print(f"   Target modules: {lora_config.target_modules}")
print(f"   Dropout: {lora_config.lora_dropout}")

In [ ]:
# Cell 6: 学習の実行

from transformers import TrainingArguments, Trainer

# 学習設定
training_args = TrainingArguments(
    output_dir="outputs/mistral_v3",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=200,              # ステップ数増加（100→200）
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    report_to="none",
    logging_dir="logs/mistral_v3"
)

print("🚀 学習を開始します...")
print(f"\n【学習設定】")
print(f"  バッチサイズ: {training_args.per_device_train_batch_size}")
print(f"  勾配累積: {training_args.gradient_accumulation_steps}")
print(f"  実効バッチサイズ: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  学習率: {training_args.learning_rate}")
print(f"  最大ステップ数: {training_args.max_steps}")
print(f"  Warmupステップ: {training_args.warmup_steps}")

# Trainerの作成
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset,
    tokenizer=tokenizer
)

# 学習実行
print("\n⏳ 学習中... (約7-10分かかります)")
train_result = trainer.train()

print("\n✅ 学習完了!")
print(f"   Final Loss: {train_result.training_loss:.4f}")
print(f"   学習時間: {train_result.metrics['train_runtime']:.1f}秒")

In [ ]:
# Cell 7: モデルの保存

OUTPUT_DIR = "models/mistral7b_v3_finetuned"

print(f"💾 モデルを保存中...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ モデル保存完了!")
print(f"   保存先: {OUTPUT_DIR}")

In [ ]:
# Cell 8: 評価用テストケースの定義

TEST_CASES = [
    {
        "category": "技術説明",
        "input": "Transformerアーキテクチャについて教えてください",
        "expected_keywords": ["attention", "アテンション", "並列", "エンコーダ", "デコーダ"]
    },
    {
        "category": "コーディング",
        "input": "Pythonでリスト内包表記を使う方法は?",
        "expected_keywords": ["リスト", "for", "if", "簡潔", "効率"]
    },
    {
        "category": "問題解決",
        "input": "モデルが過学習している場合の対策は?",
        "expected_keywords": ["dropout", "正則化", "データ", "early stopping"]
    },
    {
        "category": "比較",
        "input": "PyTorchとTensorFlowの違いは?",
        "expected_keywords": ["動的", "静的", "グラフ", "研究", "本番"]
    },
    {
        "category": "対話的",
        "input": "機械学習を勉強したいんだけど、どうすればいい?",
        "expected_keywords": ["Python", "数学", "実践", "プロジェクト", "データ"]
    },
    {
        "category": "新規質問",
        "input": "ニューラルネットワークとは何ですか?",
        "expected_keywords": ["層", "ニューロン", "重み", "学習", "予測"]
    },
    {
        "category": "新規質問",
        "input": "GitとGitHubの違いは?",
        "expected_keywords": ["バージョン", "管理", "リポジトリ", "共有", "ホスティング"]
    }
]

print("✅ テストケース準備完了")
print(f"   テストケース数: {len(TEST_CASES)}件")

In [ ]:
# Cell 9: モデルの再ロードと推論テスト

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel
import torch

print("🔄 Fine-tunedモデルをロード中...")

# ベースモデルのロード
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

# LoRAアダプターの適用
finetuned_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
finetuned_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

# パイプラインの作成
pipe = pipeline(
    "text-generation",
    model=finetuned_model,
    tokenizer=finetuned_tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1
)

print("✅ モデルロード完了!")

# 簡易テスト
print("\n【簡易推論テスト】")
test_prompt = "User: Pythonとは何ですか?\nBot:"
result = pipe(test_prompt, return_full_text=False)
response = result[0]['generated_text']

print(f"質問: Pythonとは何ですか?")
print(f"応答: {response}")

In [ ]:
# Cell 10: 評価システムの実装

import numpy as np
from typing import List, Dict
from datetime import datetime

class ModelEvaluator:
    """モデル評価クラス"""
    
    def __init__(self, pipe):
        self.pipe = pipe
    
    def evaluate_test_cases(self, test_cases: List[Dict]) -> Dict:
        """テストケースでの評価"""
        
        print("🔍 評価を開始します...\n")
        
        results = {
            "timestamp": datetime.now().isoformat(),
            "test_results": [],
            "summary": {}
        }
        
        all_keyword_scores = []
        all_response_lengths = []
        
        for i, test_case in enumerate(test_cases, 1):
            print(f"[{i}/{len(test_cases)}] {test_case['category']}: {test_case['input'][:40]}...")
            
            # 応答生成
            prompt = f"User: {test_case['input']}\nBot:"
            response = self.pipe(prompt, return_full_text=False)[0]['generated_text']
            
            # キーワードカバレッジ計算
            response_lower = response.lower()
            matched_keywords = [kw for kw in test_case['expected_keywords'] 
                              if kw.lower() in response_lower]
            keyword_score = len(matched_keywords) / len(test_case['expected_keywords'])
            
            # 応答品質チェック
            response_length = len(response.split())
            is_coherent = self._check_coherence(response)
            
            # 結果記録
            result = {
                "category": test_case['category'],
                "input": test_case['input'],
                "response": response,
                "keyword_coverage": keyword_score,
                "matched_keywords": matched_keywords,
                "response_length": response_length,
                "is_coherent": is_coherent
            }
            
            results['test_results'].append(result)
            all_keyword_scores.append(keyword_score)
            all_response_lengths.append(response_length)
            
            print(f"  ✓ キーワード: {keyword_score:.1%} | 長さ: {response_length}語 | 一貫性: {'○' if is_coherent else '×'}")
        
        # サマリー計算
        results['summary'] = {
            "avg_keyword_coverage": np.mean(all_keyword_scores),
            "avg_response_length": np.mean(all_response_lengths),
            "coherence_rate": sum(r['is_coherent'] for r in results['test_results']) / len(test_cases),
            "total_tests": len(test_cases)
        }
        
        print(f"\n{'='*60}")
        print("📊 評価完了!")
        print(f"{'='*60}")
        print(f"平均キーワードカバレッジ: {results['summary']['avg_keyword_coverage']:.1%}")
        print(f"平均応答長: {results['summary']['avg_response_length']:.1f}語")
        print(f"一貫性のある応答率: {results['summary']['coherence_rate']:.1%}")
        
        return results
    
    def _check_coherence(self, response: str) -> bool:
        """応答の一貫性をチェック"""
        # 基本的なチェック
        checks = [
            len(response.strip()) > 0,              # 空でない
            len(response.split()) >= 10,            # 最低10語以上
            not self._is_repetitive(response),      # 繰り返しでない
            not self._contains_gibberish(response)  # 意味不明でない
        ]
        return all(checks)
    
    def _is_repetitive(self, text: str) -> bool:
        """繰り返しが多いかチェック"""
        words = text.split()
        if len(words) < 10:
            return False
        unique_ratio = len(set(words)) / len(words)
        return unique_ratio < 0.3
    
    def _contains_gibberish(self, text: str) -> bool:
        """意味不明な文字列を含むかチェック"""
        import re
        return bool(re.search(r'(.)\1{4,}', text))
    
    def generate_report(self, results: Dict, filename: str = "evaluations/evaluation_report_v3.json"):
        """評価レポートを保存"""
        import json
        
        # JSON保存
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        
        # Markdown生成
        md_report = self._create_markdown_report(results)
        md_filename = filename.replace('.json', '.md')
        with open(md_filename, 'w', encoding='utf-8') as f:
            f.write(md_report)
        
        print(f"\n✅ レポート保存完了")
        print(f"  JSON: {filename}")
        print(f"  Markdown: {md_filename}")
    
    def _create_markdown_report(self, results: Dict) -> str:
        """Markdownレポート生成"""
        md = "# Mistral-7B v3 評価レポート\n\n"
        md += f"**評価日時:** {results['timestamp']}\n\n"
        
        # サマリー
        summary = results['summary']
        md += "## 📊 総合評価\n\n"
        md += f"- **キーワードカバレッジ:** {summary['avg_keyword_coverage']:.1%}\n"
        md += f"- **平均応答長:** {summary['avg_response_length']:.1f}語\n"
        md += f"- **一貫性率:** {summary['coherence_rate']:.1%}\n"
        md += f"- **テスト数:** {summary['total_tests']}件\n\n"
        
        # 詳細結果
        md += "## 📝 詳細結果\n\n"
        for result in results['test_results']:
            md += f"### {result['category']}\n\n"
            md += f"**質問:** {result['input']}\n\n"
            md += f"**応答:**\n> {result['response']}\n\n"
            md += f"- キーワードカバレッジ: {result['keyword_coverage']:.1%}\n"
            md += f"- マッチしたキーワード: {', '.join(result['matched_keywords'])}\n"
            md += f"- 応答長: {result['response_length']}語\n"
            md += f"- 一貫性: {'✓' if result['is_coherent'] else '✗'}\n\n"
            md += "---\n\n"
        
        return md

# 評価器の作成
evaluator = ModelEvaluator(pipe)

print("✅ 評価システム準備完了")

In [ ]:
# Cell 11: 評価の実行

# テストケースで評価実行
evaluation_results = evaluator.evaluate_test_cases(TEST_CASES)

# レポート生成
evaluator.generate_report(evaluation_results)

# 結果の表示
print("\n" + "="*60)
print("🎉 評価プロセス完了!")
print("="*60)

In [ ]:
# Cell 12: Gradio UIの構築

import gradio as gr

def chat_with_model(message, history):
    """チャット関数"""
    # プロンプト構築
    prompt = f"User: {message}\nBot:"
    
    # 応答生成
    response = pipe(
        prompt,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        return_full_text=False
    )[0]['generated_text']
    
    return response

# Gradio ChatInterface
demo = gr.ChatInterface(
    fn=chat_with_model,
    title="🤖 Mistral-7B v3 Fine-tuned Chatbot",
    description="""
    **LoRA Fine-tuned Mistral-7B チャットボット (v3)**
    
    改善点:
    - データセット拡張: 25件 (技術説明、コーディング、問題解決、比較、対話)
    - LoRA Rank増加: 8 → 16
    - ターゲットモジュール拡大: 4種類
    - 学習ステップ増加: 100 → 200
    
    気軽に質問してください!
    """,
    examples=[
        "Transformerアーキテクチャについて教えてください",
        "Pythonでファイルを読み込む方法は?",
        "機械学習モデルが過学習した場合の対策は?",
        "PyTorchとTensorFlowの違いは?",
        "AIについて勉強したいです。何から始めればいいですか?"
    ],
    theme=gr.themes.Soft(),
    cache_examples=False
)

print("🎨 Gradio UI準備完了")
print("\n次のセルで demo.launch() を実行してUIを起動してください")

In [ ]:
# Cell 13: Gradio UI起動

# Gradio UIの起動
demo.launch(share=True, debug=False)

In [ ]:
# Cell 14: 比較分析 - v2との比較

print("📈 v2 vs v3 比較分析\n")
print("="*60)

# 想定される改善点
comparison = {
    "項目": [
        "データセット",
        "LoRA Rank",
        "ターゲットモジュール",
        "学習ステップ",
        "予想Final Loss",
        "予想キーワードカバレッジ",
        "予想応答品質"
    ],
    "v2 (baseline)": [
        "100件",
        "8",
        "2種類 (q_proj, v_proj)",
        "100",
        "0.044",
        "50-60%",
        "中程度"
    ],
    "v3 (enhanced)": [
        "25件 (高品質)",
        "16",
        "4種類 (q_proj, k_proj, v_proj, o_proj)",
        "200",
        "0.020-0.030",
        "70-80%",
        "高品質"
    ]
}

import pandas as pd
df_comparison = pd.DataFrame(comparison)
print(df_comparison.to_string(index=False))

print("\n" + "="*60)
print("主な改善点:")
print("  1. データ品質の向上（カテゴリ別に整理）")
print("  2. モデルの表現力向上（Rank倍増）")
print("  3. より多くの注意機構を学習（4モジュール）")
print("  4. 学習の安定化（ステップ数増加）")

In [ ]:
# Cell 15: 実験ログの保存

import json
from datetime import datetime

# 実験設定のログ
experiment_log = {
    "version": "v3",
    "timestamp": datetime.now().isoformat(),
    "model": {
        "base_model": MODEL_NAME,
        "quantization": "4bit NF4",
        "lora_config": {
            "r": 16,
            "lora_alpha": 32,
            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
            "lora_dropout": 0.1
        }
    },
    "training": {
        "dataset_size": len(processed_dataset),
        "batch_size": 1,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-4,
        "max_steps": 200,
        "warmup_steps": 10
    },
    "results": {
        "final_loss": train_result.training_loss,
        "training_time_seconds": train_result.metrics['train_runtime'],
        "evaluation_summary": evaluation_results['summary']
    }
}

# ログ保存
log_filename = f"logs/experiment_log_v3_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(log_filename, 'w', encoding='utf-8') as f:
    json.dump(experiment_log, f, ensure_ascii=False, indent=2)

print(f"✅ 実験ログ保存完了: {log_filename}")

print("\n" + "="*80)
print("🎊 すべての処理が完了しました!")
print("="*80)
print("\n📁 生成されたファイル:")
print(f"  - モデル: {OUTPUT_DIR}")
print(f"  - データ: data/training_data_v3.json")
print(f"  - 評価レポート: evaluations/evaluation_report_v3.json")
print(f"  - 評価レポート(MD): evaluations/evaluation_report_v3.md")
print(f"  - 実験ログ: {log_filename}")
print("\n💡 次のステップ:")
print("  1. 評価レポートを確認")
print("  2. Gradio UIで実際に対話テスト")
print("  3. 必要に応じてハイパーパラメータ調整")
print("  4. GitHub/HuggingFaceにアップロード")

In [ ]:
# Cell 16 (Optional): 詳細な応答分析

print("📊 応答品質の詳細分析\n")

def analyze_response_quality(test_results):
    """応答品質の詳細分析"""
    
    # カテゴリ別スコア
    category_scores = {}
    for result in test_results:
        cat = result['category']
        if cat not in category_scores:
            category_scores[cat] = []
        category_scores[cat].append(result['keyword_coverage'])
    
    print("【カテゴリ別キーワードカバレッジ】")
    for cat, scores in category_scores.items():
        avg_score = np.mean(scores)
        print(f"  {cat}: {avg_score:.1%}")
    
    # 応答長の分布
    lengths = [r['response_length'] for r in test_results]
    print(f"\n【応答長の統計】")
    print(f"  平均: {np.mean(lengths):.1f}語")
    print(f"  最小: {np.min(lengths)}語")
    print(f"  最大: {np.max(lengths)}語")
    print(f"  中央値: {np.median(lengths):.1f}語")
    
    # 一貫性チェック
    coherent_count = sum(r['is_coherent'] for r in test_results)
    print(f"\n【一貫性】")
    print(f"  一貫性のある応答: {coherent_count}/{len(test_results)} ({coherent_count/len(test_results):.1%})")

# 分析実行
analyze_response_quality(evaluation_results['test_results'])

print("\n✅ 詳細分析完了")

In [ ]:
# Cell 17 (Optional): ベストプラクティス例の表示

print("💡 ファインチューニングのベストプラクティス\n")
print("="*60)

best_practices = """
1. **データ品質が最重要**
   - 量より質: 高品質な25-100件 > 低品質な1000件
   - カテゴリバランス: 偏りのないデータ分布
   - 多様性: 様々な質問パターンと応答スタイル

2. **LoRA設定の調整**
   - Rank: 4-32の範囲で実験（メモリと性能のトレードオフ）
   - Alpha: 通常はRankの2倍
   - Target Modules: より多くのモジュールで表現力↑

3. **学習パラメータ**
   - Learning Rate: 1e-4 ~ 3e-4が安全
   - Batch Size: 実効バッチサイズ4-8を推奨
   - Steps: データ量に応じて100-500

4. **評価とモニタリング**
   - 定期的な評価で過学習を検出
   - 複数の指標で総合的に判断
   - 実際の対話で定性評価も実施

5. **イテレーション**
   - 小さく始めて徐々に改善
   - A/Bテストで効果を検証
   - ログを残して再現可能に
"""

print(best_practices)

print("="*60)
print("✨ Happy Fine-tuning! ✨")